# ChromaDB 数据查看

交互式查看和分析 ChromaDB 中的向量数据

In [ ]:
import sys
from pathlib import Path

# 添加项目路径
sys.path.insert(0, str(Path.cwd().parent))

from app.core.config import settings
from app.services.chroma import get_chroma_service

In [ ]:
# 初始化 ChromaDB
chroma = get_chroma_service()
print(f"存储路径: {settings.CHROMA_PERSIST_DIR}")

In [ ]:
# 列出所有 collection
collections = chroma.client.list_collections()
print(f"总数: {len(collections)}\n")

for coll in collections:
    print(f"📚 {coll.name} - {coll.count()} 条向量")

In [ ]:
# 查看某个知识库的数据（修改 kb_id）
kb_id = 1
collection = chroma.get_collection(kb_id)

# 获取所有数据
result = collection.get(include=["documents", "metadatas", "embeddings"])

print(f"知识库 {kb_id} 数据:")
print(f"向量数: {len(result['ids'])}")
print(f"\n前 5 条:")

for i in range(min(5, len(result['ids']))):
    print(f"\n[{i+1}] {result['ids'][i]}")
    print(f"文档: {result['documents'][i][:200]}...")
    print(f"元数据: {result['metadatas'][i]}")
    print(f"向量维度: {len(result['embeddings'][i])}")

In [ ]:
# 语义搜索测试
from app.services.embedding import get_embedding_service

query = "你的搜索问题"
embedding_service = get_embedding_service()
query_embedding = embedding_service.encode_single(query)

results = collection.query(
    query_embeddings=[query_embedding],
    n_results=5,
    include=["documents", "metadatas", "distances"]
)

print(f"搜索: {query}\n")
for i, (doc, meta, dist) in enumerate(zip(
    results['documents'][0],
    results['metadatas'][0],
    results['distances'][0]
), 1):
    print(f"[{i}] 相似度: {1 - dist:.4f}")
    print(f"    {doc[:150]}...")
    print(f"    {meta}\n")